# Selected Samples Extraction Workbench v6

Notebook-only extraction workbench for `samples/selected samples/`.

This version refines the issues found in v5:

- false abnormal flags
- wrong doctor names
- missing image common fields
- OCR-gibberish detection
- relative output paths
- privacy / Git tracking
- clean side-by-side comparison with source

Outputs:

```text
notebook_outputs/selected_samples_v6/
  document_summary.csv
  document_json_rows.csv
  documents.jsonl
  extracted_parts.csv
  extracted_texts.csv
  review_compare.html
  privacy_report.json
  texts/
  sanitized_texts/
  json/
  source_previews/
  debug/
```


In [12]:
# =========================
# 0) Setup
# =========================
from pathlib import Path
import os, re, json, math, hashlib, traceback, html, unicodedata
from dataclasses import dataclass, asdict, field
from collections import defaultdict
import pandas as pd
import numpy as np

try:
    from PIL import Image, ImageOps, ImageEnhance, ImageFilter
except Exception as e:
    raise RuntimeError('Install Pillow: pip install pillow') from e

try:
    import fitz
except Exception:
    fitz = None

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    TESSERACT_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p/'app').exists() or (p/'samples').exists() or (p/'.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root()
SELECTED_DIR = REPO_ROOT / 'samples' / 'selected samples'
OUTPUT_DIR = REPO_ROOT / 'notebook_outputs' / 'selected_samples_v6'
TEXT_DIR = OUTPUT_DIR / 'texts'
SANITIZED_TEXT_DIR = OUTPUT_DIR / 'sanitized_texts'
JSON_DIR = OUTPUT_DIR / 'json'
PREVIEW_DIR = OUTPUT_DIR / 'source_previews'
DEBUG_DIR = OUTPUT_DIR / 'debug'
VARIANT_TEXT_DIR = DEBUG_DIR / 'ocr_variants'
DEBUG_IMAGE_DIR = DEBUG_DIR / 'images'
for d in [OUTPUT_DIR, TEXT_DIR, SANITIZED_TEXT_DIR, JSON_DIR, PREVIEW_DIR, DEBUG_DIR, VARIANT_TEXT_DIR, DEBUG_IMAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {'.pdf','.jpg','.jpeg','.png','.webp','.tif','.tiff'}
print('Repo root:', REPO_ROOT)
print('Selected dir:', SELECTED_DIR)
print('Output dir:', OUTPUT_DIR)
print('Tesseract:', TESSERACT_AVAILABLE, '| OpenCV:', CV2_AVAILABLE, '| PyMuPDF:', fitz is not None)


Repo root: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents
Selected dir: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/samples/selected samples
Output dir: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selected_samples_v6
Tesseract: True | OpenCV: True | PyMuPDF: True


In [2]:
# =========================
# 1) Privacy / Git tracking guard
# =========================
PRIVACY_MODE = 'internal'  # 'internal' keeps exact local OCR text; 'safe_share' masks text inside CSV/JSON columns
AUTO_UPDATE_GITIGNORE = False
RECOMMENDED_GITIGNORE_LINES = [
    'samples/selected samples/',
    'samples/selected_samples/',
    'notebook_outputs/',
    '*.db',
    'eval_storage/',
    'debug_output/',
]

def relpath(p):
    try:
        return str(Path(p).resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(p)

def check_gitignore():
    gi = REPO_ROOT / '.gitignore'
    txt = gi.read_text(encoding='utf-8') if gi.exists() else ''
    missing = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in txt]
    if AUTO_UPDATE_GITIGNORE and missing:
        with gi.open('a', encoding='utf-8') as f:
            f.write('\n# Notebook PHI/PII and debug outputs\n')
            for line in missing:
                f.write(line + '\n')
        txt = gi.read_text(encoding='utf-8')
        missing = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in txt]
    return {'gitignore_path': relpath(gi), 'exists': gi.exists(), 'missing_recommended_lines': missing, 'auto_updated': AUTO_UPDATE_GITIGNORE and not missing}

privacy_report = check_gitignore()
print(json.dumps(privacy_report, ensure_ascii=False, indent=2))
(OUTPUT_DIR/'privacy_report.json').write_text(json.dumps(privacy_report, ensure_ascii=False, indent=2), encoding='utf-8')


{
  "gitignore_path": ".gitignore",
  "exists": true,
  "missing_recommended_lines": [
    "samples/selected samples/",
    "samples/selected_samples/",
    "notebook_outputs/"
  ],
  "auto_updated": false
}


207

In [14]:
# =========================
# 2) Discover selected files
# =========================
ONLY_FILES = []  # Optional list of paths. Leave empty to scan selected folder.

def discover_files():
    if ONLY_FILES:
        out = []
        for x in ONLY_FILES:
            p = Path(x)
            if not p.is_absolute(): p = REPO_ROOT / p
            if p.exists() and p.is_file() and p.suffix.lower() in SUPPORTED_EXTS:
                out.append(p.resolve())
        return out
    #folders = [SELECTED_DIR, REPO_ROOT/'samples'/'selected_samples', REPO_ROOT/'samples'/'raw']
    folders = [SELECTED_DIR]
    found = []
    for folder in folders:
        if folder.exists():
            for ext in SUPPORTED_EXTS:
                found += list(folder.rglob(f'*{ext}')) + list(folder.rglob(f'*{ext.upper()}'))
    seen, clean = set(), []
    for p in found:
        rp = p.resolve()
        if rp.is_file() and rp not in seen:
            seen.add(rp); clean.append(rp)
    return sorted(clean, key=lambda p: p.name)

FILES = discover_files()
print(f'Found {len(FILES)} files')
for i,p in enumerate(FILES,1): print(f'{i:03d} | {p.name} | {p.suffix.lower()} | {p.stat().st_size/1024:.1f} KB | {relpath(p)}')
assert FILES, 'No files found. Put samples in samples/selected samples/ or set ONLY_FILES.'


Found 20 files
001 | -2147483648_-210193.jpg | .jpg | 77.8 KB | samples/selected samples/-2147483648_-210193.jpg
002 | -2147483648_-210195.jpg | .jpg | 42.0 KB | samples/selected samples/-2147483648_-210195.jpg
003 | 0014161672_14041209_O_00404121721.pdf | .pdf | 44.2 KB | samples/selected samples/0014161672_14041209_O_00404121721.pdf
004 | 0020139871_14041209_O_00404121714.pdf | .pdf | 41.4 KB | samples/selected samples/0020139871_14041209_O_00404121714.pdf
005 | 0021858845_14041209_O_00404121731.pdf | .pdf | 42.9 KB | samples/selected samples/0021858845_14041209_O_00404121731.pdf
006 | 0023471026_14041209_O_00404121728.pdf | .pdf | 40.8 KB | samples/selected samples/0023471026_14041209_O_00404121728.pdf
007 | 0024150010_14041209_O_00404121722.pdf | .pdf | 44.1 KB | samples/selected samples/0024150010_14041209_O_00404121722.pdf
008 | 0025631314_14041209_O_00404121726.pdf | .pdf | 41.2 KB | samples/selected samples/0025631314_14041209_O_00404121726.pdf
009 | 0025692283_14041209_O_00404

In [15]:
# =========================
# 3) Config and dictionaries
# =========================
CONFIG = {
    'max_pdf_pages': 5,
    'min_ocr_text_len': 20,
    'good_quality_min_score': 0.60,
    'poor_quality_min_score': 0.22,
    'min_good_text_quality': 0.55,
    'min_usable_text_quality': 0.28,
    'save_variant_texts': True,
    'save_debug_images': True,
    'run_rotation_variants': True,
    'psm_modes': [6,4,11],
    'tesseract_langs': ['eng+fas','eng'],
    'source_preview_width': 900,
}

MEDICAL_KEYWORDS = ['laboratory','lab','hematology','biochemistry','serology','hormone','thyroid','urine','culture','bacteriology','cbc','wbc','rbc','hemoglobin','platelet','fbs','glucose','creatinine','cholesterol','triglycerides','tsh','ferritin','vitamin','crp','pt','inr','ptt','reference range','method','آزمایشگاه','آزمايشگاه','پاتوبیولوژی','پاتوبيولوژی','درمانگاه','بیمارستان','پزشک','پذیرش','نتیجه','ادرار','خون','نام بیمار','نام مراجعه']

LAB_ALIASES = {
 'WBC':['WBC','W.B.C','White Blood Cell'], 'RBC':['RBC','R.B.C','Red Blood Cell'], 'HGB':['HGB','Hb','Hemoglobin','Haemoglobin','Hemoglobine'], 'HCT':['HCT','H.C.T','Hematocrit','Het'], 'MCV':['MCV','M.C.V'], 'MCH':['MCH','M.C.H'], 'MCHC':['MCHC','M.C.H.C'], 'RDW-CV':['RDW-CV','RDW CV','RDW.'], 'RDW-SD':['RDW-SD','RDW SD'], 'PLT':['PLT','Platelet','Platelets'], 'PDW':['PDW'], 'MPV':['MPV'], 'P-LCR':['P-LCR','PLCR'], 'PCT':['PCT'], 'NEUT#':['NEUT#','Neut#','Neutrophil#'], 'LYMPH#':['LYMPH#','Lymph#','Lymphocyte#'], 'MONO#':['MONO#','Mono#','Monocyte#'], 'EO#':['EO#','EOS#','Eosinophil#'], 'BASO#':['BASO#','Baso#','Basophil#'], 'NEUT%':['NEUT%','Neut%','Neutrophils','Neutrophil'], 'LYMPH%':['LYMPH%','Lymph%','Lymphocytes','Lymphocyte','Lympho'], 'MONO%':['MONO%','Mono%','Monocytes','Monocyte'], 'EO%':['EO%','EOS%','Eosinophils','Eosinophil'], 'BASO%':['BASO%','Basophils','Basophil'], 'ESR':['ESR','Erythrocyte Sediment Rate','Erythrocyte Sedimentation Rate'],
 'FBS':['Fasting Blood Glucose','Fasting blood sugar','Fasting blood segar','FBS','Glucose','Fasting Serum Glucose','Blood sugar'], 'HbA1c':['HbA1c','Hemoglobin A1c','Hemoglobine A1c','Hemoglobin Ate'], 'EAG':['Estimated Average Glucose','EAG','EAS'], 'BUN':['BUN','Blood Urea Nitrogen','Blood Urea nitrogen','Serum Urea Nitrogen'], 'Urea':['Urea'], 'Creatinine':['Creatinine','Creatininee','Creatinin'], 'Uric Acid':['Uric Acid'], 'Total Cholesterol':['Total Cholesterol','Cholesterol','Cholestrol','Cholestrol-Total'], 'Triglycerides':['Triglycerides','Triglycerideses','Triglycerid','TG'], 'HDL':['HDL','HDL Cholesterol','HDL-Cholesterol'], 'LDL':['LDL','LDL Cholesterol','LDL-Cholesterol','LDL - Cholesterol','LDL-Cholesterol (Direct)'], 'AST':['AST','SGOT','SGOT(AST)','AST(SGOT)'], 'ALT':['ALT','SGPT','SGPT(ALT)'], 'ALP':['ALP','Alkaline Phosphatase'], 'Calcium':['Calcium','Calcium (Ca)'], 'Phosphorus':['Phosphorus','P'], 'Sodium':['Sodium','Na'], 'Potassium':['Potassium','K'], 'Iron':['Iron','Fe'], 'TIBC':['TIBC','Total Iron Binding Capacity'], 'Bilirubin Total':['Bilirubin Total','Bilirubin(Total)'], 'Bilirubin Direct':['Bilirubin Direct','Bilirubin(Direct)'],
 'TSH':['TSH','Thyroid Stimulating Hormone','Simulating Hormone'], 'T3':['T3','Tri iodothyronine','Triiodothyronine','Tri Jodordyronine'], 'T4':['T4','Thyroxine'], 'Free T4':['Free T4','FT4','Free 14'], 'Vitamin D':['Vitamin D','25 Hydroxy Vitamin D','25-Hydroxy Vitamin D3','25-OH Vitamin D3','25:2ydrovy Vieamin Ds'], 'Vitamin B12':['Vitamin B12'], 'Ferritin':['Ferritin'], 'CRP':['CRP','C-Reactive protein','C - Reactive protein'], 'LH':['LH'], 'FSH':['FSH'], 'Prolactin':['Prolactin','Protactin'], 'Testosterone':['Testosterone'], 'Free Testosterone':['Free Testosterone'], 'Estradiol':['Estradiol','Estradiot'], 'DHEA-SO4':['DHEA-SO4','DHEA SO4'], '17OH-Progesterone':['17OH-Progesterone','17OH Progesterone','ow-Progesterone'],
 'PT':['PT','Prothrombin Time'], 'PT Control':['PT Control','PT Controt'], 'INR':['INR'], 'PTT':['PTT','Activated PTT','Activated PT'],
 'Urine Culture':['Urine Culture','Urine Cultures','Culture'], 'Color':['Color','olor'], 'Appearance':['Appearance','Appenrance'], 'Specific Gravity':['Specific Gravity','Sp. Gravity','Specie Gravity'], 'pH':['pH','PH'], 'Protein':['Protein'], 'Urine Glucose':['Urine Glucose','Glucose'], 'Ketone':['Ketone'], 'Bilirubin Urine':['Bilirubin'], 'Urobilinogen':['Urobilinogen','Uobinogen','Urobitinogen'], 'Nitrite':['Nitrite','Nirive'], 'Blood/Hb':['Blood/Hb','Hemoglobin','Blood','Blooato'], 'WBC/HPF':['WBC/HPF','W.B.C / HPF','W.B.C HPF','wncsnpe'], 'RBC/HPF':['RBC/HPF','R.B.C / HPF','R.B.C HPF'], 'Epithelial Cells':['Epithelial Cells','Epithelial Cell','plthetal cells'], 'Bacteria':['Bacteria','acterta'], 'Mucus':['Mucus'], 'Casts':['Casts','Cast'], 'Crystals':['Crystals','Crystal'],
}
ALL_ALIAS_PHRASES = [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases]
GUIDELINE_WORDS_RE = re.compile(r'\b(Normal|Borderline|Bordreline|Borrderline|Desirable|High risk|Low risk|Adults?|Males?|Females?|Women|Men|Pregnant|Below age|Above age|Reference|Range|Prognose|Abnormal|Doubtful)\b', re.I)


In [16]:
# =========================
# 4) Normalization, sanitization, quality, source preview
# =========================
DIGIT_MAP = str.maketrans('۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789')

def normalize_digits(text): return (text or '').translate(DIGIT_MAP)

def normalize_text(text):
    text = unicodedata.normalize('NFKC', text or '')
    text = normalize_digits(text)
    text = text.replace('ي','ی').replace('ك','ک').replace('ۀ','ه')
    corr = {'خائم':'خانم','خانمز':'خانم ز','خانمزهرا':'خانم زهرا','خانمسیده':'خانم سیده','بیمار نام':'نام بیمار','پزشک نام':'نام پزشک','پذیرش تاریخ':'تاریخ پذیرش','پدیرش':'پذیرش','پذبرش':'پذیرش','پذيرش':'پذیرش','تاريخ':'تاریخ','دكتر':'دکتر','پزشك':'پزشک','معالح':'معالج','Refrence':'Reference','Aeference':'Reference','Ranwe':'Range','Kefecence':'Reference','Methad':'Method','Fasting Blood Glucase':'Fasting Blood Glucose','Fasting blood segar':'Fasting blood sugar','Hemoglobine Alc':'Hemoglobin A1c','Hemoglobin Ate':'Hemoglobin A1c','Cholestrol':'Cholesterol','Triglycerideses':'Triglycerides','Creatininee':'Creatinine','Creatinin':'Creatinine'}
    for a,b in corr.items(): text = text.replace(a,b)
    text = re.sub(r'[^\S\n]+',' ', text)
    return re.sub(r'\n{3,}','\n\n', text).strip()

def clean_value(v, max_len=120):
    if not v: return None
    v = normalize_text(v)
    v = re.sub(r'^[\s:：|#*/\\.,؛;-]+|[\s:：|#*/\\.,؛;-]+$', '', v).strip()
    v = re.sub(r'\s+', ' ', v)
    return v if v and len(v) <= max_len else None

def safe_id(path):
    stem = re.sub(r'[^\w\u0600-\u06FF.-]+', '_', path.stem)
    return stem[:80] or hashlib.md5(str(path).encode()).hexdigest()[:10]

def hash_value(v): return hashlib.sha256(str(v).encode('utf-8')).hexdigest() if v else None

def sanitize_text(text):
    t = text or ''
    t = re.sub(r'(?<!\d)\d{10}(?!\d)', '**********', t)
    t = re.sub(r'(?<!\d)09\d{9}(?!\d)', '09*********', t)
    return t

def validate_jalali_date(date):
    if not date: return False
    m = re.match(r'^(13[9][0-9]|14[0-1][0-9]|1420)[/-](\d{1,2})[/-](\d{1,2})$', normalize_digits(date))
    if not m: return False
    y,mo,d = map(int, m.groups())
    return 1390 <= y <= 1420 and 1 <= mo <= 12 and 1 <= d <= 31

def file_mime_guess(path):
    return {'.pdf':'application/pdf','.jpg':'image/jpeg','.jpeg':'image/jpeg','.png':'image/png','.webp':'image/webp','.tif':'image/tiff','.tiff':'image/tiff'}.get(path.suffix.lower(),'application/octet-stream')

def assess_image_quality(path):
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
        w,h = img.size; gray = np.array(img.convert('L'))
        mean, std = float(np.mean(gray)), float(np.std(gray))
        lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if CV2_AVAILABLE else 0.0
        blur = min(1.0, lap/500.0) if lap else 0.5; bright = max(0, 1-abs(mean-127)/127); contrast = min(1, std/80); res = min(1, (w*h)/(1000*1000))
        score = blur*.35 + bright*.20 + contrast*.25 + res*.20
        issues=[]
        if lap and blur < .18: issues.append('severe_blur')
        if contrast < .25: issues.append('low_contrast')
        if mean < 45: issues.append('too_dark')
        if mean > 220: issues.append('too_bright')
        if res < .10: issues.append('low_resolution')
        if h > w*1.35 or w > h*1.35: issues.append('possible_rotation_or_sideways')
        status = 'good_quality' if score >= CONFIG['good_quality_min_score'] else ('needs_preprocessing' if score >= CONFIG['poor_quality_min_score'] else 'poor_quality')
        return {'status':status,'overall_quality_score':round(score,3),'issues':issues,'width':w,'height':h,'brightness_mean':round(mean,2),'contrast_std':round(std,2),'laplacian_variance':round(lap,2)}
    except Exception as e:
        return {'status':'unreadable','overall_quality_score':0.0,'issues':['unreadable_image',str(e)]}

def assess_file_quality(path):
    return {'status':'pdf_quality_not_assessed_here','overall_quality_score':None,'issues':[],'width':None,'height':None} if path.suffix.lower()=='.pdf' else assess_image_quality(path)

def create_source_preview(path, sid):
    out = PREVIEW_DIR / f'{sid}.png'
    try:
        if path.suffix.lower()=='.pdf' and fitz is not None:
            doc=fitz.open(path); pix=doc[0].get_pixmap(matrix=fitz.Matrix(2,2), alpha=False); pix.save(str(out)); doc.close()
        else:
            img=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); img.thumbnail((CONFIG['source_preview_width'], CONFIG['source_preview_width'])); img.save(out)
        return relpath(out)
    except Exception:
        return None


In [17]:
# =========================
# 5) OCR variants and OCR text quality
# =========================
@dataclass
class OCRCandidate:
    text: str; confidence: float; variant: str; psm: int|None; lang: str; score: float; error: str|None=None; details: dict=field(default_factory=dict)

def reconstruct_lines_from_tesseract_data(data):
    rows=[]; n=len(data.get('text',[]))
    for i in range(n):
        txt=str(data['text'][i] or '').strip()
        if not txt: continue
        try: conf=float(data.get('conf',['-1'])[i])
        except Exception: conf=-1
        if conf < 0: continue
        key=(int(data.get('block_num',[0]*n)[i]), int(data.get('par_num',[0]*n)[i]), int(data.get('line_num',[0]*n)[i]))
        rows.append({'key':key,'text':txt,'conf':conf,'left':int(data.get('left',[0]*n)[i]),'top':int(data.get('top',[0]*n)[i])})
    groups=defaultdict(list)
    for r in rows: groups[r['key']].append(r)
    lines=[]; confs=[]
    for _,items in groups.items():
        items=sorted(items, key=lambda x:x['left']); line=' '.join(x['text'] for x in items).strip()
        if line:
            lines.append((min(x['top'] for x in items), line)); confs += [x['conf'] for x in items]
    lines=sorted(lines, key=lambda x:x[0])
    return '\n'.join(line for _,line in lines), (float(np.mean(confs)/100) if confs else 0.0), {'line_count':len(lines),'word_count':len(rows)}

def image_variants(path, sid):
    img0 = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    vars=[('original_oriented',img0)]
    gray=ImageOps.grayscale(img0)
    vars.append(('light_autocontrast', ImageOps.autocontrast(gray).convert('RGB')))
    vars.append(('mild_contrast', ImageOps.autocontrast(ImageEnhance.Contrast(gray).enhance(1.4)).convert('RGB')))
    vars.append(('light_sharpen', ImageOps.autocontrast(gray.filter(ImageFilter.SHARPEN)).convert('RGB')))
    if CV2_AVAILABLE:
        arr=np.array(gray); thr=cv2.adaptiveThreshold(arr,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11); vars.append(('threshold_fallback', Image.fromarray(thr).convert('RGB')))
    if CONFIG['run_rotation_variants']:
        for name, im in list(vars[:4]):
            for deg in [90,180,270]: vars.append((f'{name}_rot{deg}', im.rotate(deg, expand=True)))
    if CONFIG['save_debug_images']:
        folder=DEBUG_IMAGE_DIR/sid; folder.mkdir(parents=True, exist_ok=True)
        for name, im in vars: im.save(folder/f'{name}.png')
    return vars

def ocr_one_image_variant(img, lang, psm):
    if not TESSERACT_AVAILABLE: return '',0.0,{'error':'pytesseract_not_available'}
    try:
        data=pytesseract.image_to_data(img, lang=lang, config=f'--psm {psm}', output_type=pytesseract.Output.DICT)
        text,conf,meta=reconstruct_lines_from_tesseract_data(data)
        if len(text.strip()) < 10:
            fb=pytesseract.image_to_string(img, lang=lang, config=f'--psm {psm}')
            if len(fb.strip()) > len(text.strip()): text=fb; meta['fallback_image_to_string']=True
        return normalize_text(text), conf, meta
    except Exception as e:
        return '',0.0,{'error':str(e)}

def count_alias_hits(text):
    low=normalize_text(text).lower()
    return sum(1 for _,alias in ALL_ALIAS_PHRASES if alias.lower() in low)

def medical_hit_count(text):
    low=normalize_text(text).lower()
    return sum(1 for k in MEDICAL_KEYWORDS if normalize_text(k).lower() in low)

def ocr_text_quality(text, confidence=0.0):
    t=normalize_text(text); toks=re.findall(r'[A-Za-z\u0600-\u06FF0-9]+', t)
    if not toks: return {'score':0.0,'status':'empty_text','reasons':['no_tokens']}
    alias_hits=count_alias_hits(t); med_hits=medical_hit_count(t); nums=len(re.findall(r'(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)', t)); lines=len([l for l in t.splitlines() if l.strip()])
    long_gib=sum(1 for tok in toks if len(tok)>=12 and not re.search(r'[aeiouآایو]', tok, re.I)); gib=long_gib/max(1,len(toks))
    score=min(len(t),3000)/3000*.20 + min(alias_hits,20)/20*.25 + min(med_hits,15)/15*.20 + min(nums,80)/80*.15 + min(lines,40)/40*.10 + min(confidence,1)*.10 - min(gib,.5)*.30
    status='good_text' if score >= CONFIG['min_good_text_quality'] else ('usable_noisy_text' if score >= CONFIG['min_usable_text_quality'] and len(t)>=CONFIG['min_ocr_text_len'] else 'gibberish_text')
    return {'score':round(max(0,float(score)),3),'status':status,'alias_hits':alias_hits,'medical_hits':med_hits,'numeric_hits':nums,'line_count':lines,'token_count':len(toks),'gibberish_ratio':round(gib,3)}

def ocr_score(text, conf): return ocr_text_quality(text, conf)['score']*100 + min(len(text),3000)/1000

def run_ocr_variants_for_image(path, sid):
    cands=[]
    for name,img in image_variants(path, sid):
        for lang in CONFIG['tesseract_langs']:
            for psm in CONFIG['psm_modes']:
                text,conf,meta=ocr_one_image_variant(img, lang, psm); cands.append(OCRCandidate(text,conf,name,psm,lang,ocr_score(text,conf),meta.get('error'),meta))
    valid=[c for c in cands if c.text and c.text.strip()]
    best=max(valid or cands, key=lambda c:c.score)
    if CONFIG['save_variant_texts']:
        folder=VARIANT_TEXT_DIR/sid; folder.mkdir(parents=True, exist_ok=True)
        for i,c in enumerate(sorted(cands, key=lambda x:x.score, reverse=True)[:25]): (folder/f'{i:02d}_{c.variant}_psm{c.psm}_{c.lang}_score{c.score:.2f}.txt').write_text(c.text or f'ERROR: {c.error}', encoding='utf-8')
    return best,cands


In [18]:
# =========================
# 6) Extract text from file
# =========================
def extract_pdf_text_layer(path):
    if fitz is None: return '',[],{'error':'pymupdf_not_available'}
    try:
        doc=fitz.open(path); pages=[normalize_text(p.get_text() or '') for p in doc[:CONFIG['max_pdf_pages']]]; meta={'page_count':doc.page_count,'used_pages':len(pages),'pdf_type':'text_pdf' if any(p.strip() for p in pages) else 'scanned_pdf'}; doc.close(); return '\f'.join(pages),pages,meta
    except Exception as e: return '',[],{'error':str(e)}

def render_pdf_pages(path, sid):
    out=[]
    if fitz is None: return out
    folder=DEBUG_IMAGE_DIR/sid/'pdf_pages'; folder.mkdir(parents=True, exist_ok=True); doc=fitz.open(path)
    for i,p in enumerate(doc[:CONFIG['max_pdf_pages']],1):
        fp=folder/f'page_{i:03d}.png'; p.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False).save(str(fp)); out.append(fp)
    doc.close(); return out

def extract_text_from_file(path, sid):
    if path.suffix.lower()=='.pdf':
        text,pages,meta=extract_pdf_text_layer(path)
        if len(text.strip()) >= CONFIG['min_ocr_text_len']:
            return {'success':True,'text':text,'pages':pages,'confidence':.95,'source':'pdf_text_layer','best_variant':'pdf_text_layer','best_psm':None,'best_lang':None,'candidate_count':0,'candidate_scores':[],'text_quality':ocr_text_quality(text,.95),'meta':meta}
        rendered=render_pdf_pages(path,sid); page_texts=[]; bests=[]
        for i,img in enumerate(rendered,1):
            best,cands=run_ocr_variants_for_image(img,f'{sid}_pdfpage{i}'); page_texts.append(best.text); bests.append(best)
        full='\f'.join(page_texts); avg=float(np.mean([b.confidence for b in bests])) if bests else 0.0
        return {'success':bool(full.strip()),'text':full,'pages':page_texts,'confidence':avg,'source':'pdf_rendered_ocr','best_variant':';'.join(b.variant for b in bests),'best_psm':';'.join(str(b.psm) for b in bests),'best_lang':';'.join(b.lang for b in bests),'candidate_count':len(bests),'candidate_scores':[asdict(b) for b in bests],'text_quality':ocr_text_quality(full,avg),'meta':meta}
    best,cands=run_ocr_variants_for_image(path,sid)
    return {'success':bool(best.text.strip()),'text':best.text,'pages':[best.text],'confidence':best.confidence,'source':'image_ocr_variants','best_variant':best.variant,'best_psm':best.psm,'best_lang':best.lang,'candidate_count':len(cands),'candidate_scores':[{'variant':c.variant,'psm':c.psm,'lang':c.lang,'score':round(c.score,2),'len':len(c.text or ''),'conf':round(c.confidence,3),'error':c.error} for c in sorted(cands, key=lambda x:x.score, reverse=True)[:20]],'text_quality':ocr_text_quality(best.text,best.confidence),'meta':{}}


In [19]:
# =========================
# 7) Classification and common fields
# =========================
TABLE_TOKENS_RE=re.compile(r'\b(Result|Unit|Reference|Range|WBC|RBC|FBS|TSH|CBC|Biochemistry|Hematology|Platelet|Creatinine|Glucose|Method)\b', re.I)

def is_medical_text(text):
    low=normalize_text(text).lower(); hits=[]
    for k in MEDICAL_KEYWORDS:
        nk=normalize_text(k).lower()
        if nk and nk in low and k not in hits: hits.append(k)
    return len(hits)/4 >= .15, hits, min(1.0,len(hits)/4)

def classify_document(text):
    t=normalize_text(text); low=t.lower(); signals={'lab':['laboratory','lab','hematology','biochemistry','serology','urine','culture','cbc','wbc','rbc','fbs','glucose','tsh','آزمایشگاه','آزمايشگاه','درمانگاه'], 'radiology':['radiology','ultrasound','sonography','mri','ct','x-ray','findings','impression','سونوگرافی','سونوگرافي','رادیولوژی'], 'pap_smear':['pap smear','hpv','nilm','ascus','asc-us','lsil','hsil','پاپ اسمیر']}
    best=('unknown_medical',0.0,[])
    for dt,sigs in signals.items():
        found=[s for s in sigs if normalize_text(s).lower() in low]
        if len(found)>len(best[2]): best=(dt,min(1.0,len(found)/5),found)
    return best

def reject_name_candidate(v):
    return (not v) or len(v)>80 or TABLE_TOKENS_RE.search(v) or len(re.findall(r'\d',v))>2 or re.search(r'پزشک|دکتر|Doctor|Result|Reference|Unit|Method', v, re.I)

def extract_tav_header(full):
    m=re.search(r'([^\n]{1,60}?)-\s*(آقای|آقاي|خانم)\s+([^\n]{1,40}?)-\s*دکتر\s*([0-9]{1,3})\s*[:：]\s*سن', full)
    if not m: return {}
    family=clean_value(m.group(1),50); given=clean_value(m.group(3),50); sex='female' if m.group(2)=='خانم' else 'male'; age=int(m.group(4)); name=clean_value(f'{given} {family}',80)
    return {'patient_name':(name,.88,m.group(0)),'sex':(sex,.9,m.group(0)),'age':(age,.9,m.group(0))}

def extract_patient_from_reversed_persian(full):
    m=re.search(r'([آ-یA-Za-z\s]{2,70}?(?:خانم|آقای|آقاي)[آ-یA-Za-z\s]{0,60})\s*(?:بیمار|بيمار)\s*نام', full)
    if m:
        raw=clean_value(m.group(1),90); sex='female' if raw and 'خانم' in raw else ('male' if raw and ('آقای' in raw or 'آقاي' in raw) else None); name=clean_value((raw or '').replace('خانم','').replace('آقای','').replace('آقاي',''),80)
        if not reject_name_candidate(name): return name,sex,m.group(0)
    return None,None,None

def extract_patient_from_compact(full):
    m=re.search(r'(?:[0-9]{1,3}\s*سال\s*)?(خانم|آقای|آقاي)\s*([آ-ی]{2,40}(?:\s*[آ-ی]{2,40}){0,3})\s*[:：]?\s*نام', full)
    if m:
        sex='female' if m.group(1)=='خانم' else 'male'; name=clean_value(m.group(2),80)
        if not reject_name_candidate(name): return name,sex,m.group(0)
    return None,None,None

def find_labeled_date(full):
    pats=[r'(?:تاریخ\s*پذیرش|تاریخ\s*جواب|تاریخ\s*گزارش|Report\s*Date|Date)\s*[:：]?\s*(?:\d{1,2}:\d{2}:\d{2}\s*[-–]?\s*)?([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})', r'([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})\s*(?:پذیرش|تاریخ)', r'(?:پذیرش|تاریخ)\s*[:：]?\s*([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})']
    for pat in pats:
        for m in re.finditer(pat,full,re.I):
            cand=normalize_digits(m.group(1))
            if validate_jalali_date(cand): return cand,m.group(0)
    return None,None

def extract_common_fields(text):
    full=normalize_text(text); lines=[l.strip() for l in full.splitlines() if l.strip()]; top='\n'.join(lines[:18])
    center=center_src=None
    for l in lines[:10]:
        if any(k in l for k in ['آزمایشگاه','آزمايشگاه','Laboratory','درمانگاه','بیمارستان','Nobin']) and not TABLE_TOKENS_RE.search(l) and len(l)<=140:
            center,center_src=clean_value(l,140),l; break
    nid=nid_src=None
    for pat in [r'(?:کد\s*ملی|کد\s*ملي|كد\s*ملي|National\s*ID)\s*[:：]?\s*([0-9]{10})', r'\b([0-9]{10})\b']:
        m=re.search(pat, full, re.I)
        if m: nid,nid_src=m.group(1),m.group(0); break
    tracking=tracking_src=None
    for pat in [r'\b([A-Za-z]-\d{5}-\d{3,})\b', r'(?:پذیرش|Admission|Report\s*No|شماره)\s*[:：]?\s*([A-Za-z0-9_-]{3,})']:
        m=re.search(pat, full, re.I)
        if m: tracking,tracking_src=m.group(1),m.group(0); break
    date,date_src=find_labeled_date(full); tav=extract_tav_header(full)
    patient_name=sex=age=None; patient_src=sex_src=age_src=None
    if 'patient_name' in tav: patient_name,_,patient_src=tav['patient_name']
    if 'sex' in tav: sex,_,sex_src=tav['sex']
    if 'age' in tav: age,_,age_src=tav['age']
    if not patient_name:
        for pat in [r'(?:نام\s*بیمار|نام\s*بيمار|نام\s*مراجعه\s*کننده)\s*[:：]?\s*(.{2,80}?)(?=\s*(?:تاریخ|سن|شماره|پزشک|کد|پذیرش|$))', r'(?:Patient\s*Name|Name)\s*[:：]\s*([^\n]{2,80})']:
            m=re.search(pat, full, re.I|re.S)
            if m:
                cand=clean_value(m.group(1),80)
                if not reject_name_candidate(cand): patient_name,patient_src=cand,m.group(0); break
    if not patient_name:
        cand,sx,src=extract_patient_from_reversed_persian(full)
        if cand: patient_name,patient_src=cand,src; sex,sex_src=(sex or sx),(sex_src or src)
    if not patient_name:
        cand,sx,src=extract_patient_from_compact(full)
        if cand: patient_name,patient_src=cand,src; sex,sex_src=(sex or sx),(sex_src or src)
    if not sex and patient_src:
        if 'خانم' in patient_src or re.search(r'\bFemale\b', patient_src, re.I): sex,sex_src='female',patient_src
        elif 'آقای' in patient_src or 'آقاي' in patient_src or re.search(r'\bMale\b', patient_src, re.I): sex,sex_src='male',patient_src
    if age is None:
        for pat in [r'سن\s*[:：]?\s*([0-9]{1,3})(?:\s*سال)?', r'([0-9]{1,3})\s*سال']:
            for m in re.finditer(pat, top, re.I):
                if re.search(r'Below age|Above age|Reference|Adults|Children|years?:', m.group(0), re.I): continue
                val=int(m.group(1))
                if 0<=val<=120: age,age_src=val,m.group(0); break
            if age is not None: break
    doctor=doctor_src=None
    for pat in [r'(?:نام\s*پزشک|پزشک\s*معالج|پزشک|Doctor|Physician)\s*[:：]?\s*([^\n]{2,80})']:
        m=re.search(pat, top, re.I)
        if m:
            cand=clean_value(m.group(1),80)
            if cand and not TABLE_TOKENS_RE.search(cand) and 'سن' not in cand and not re.search(r'دکتر\s*[0-9]{1,3}',cand): doctor,doctor_src=cand,m.group(0); break
    def field(value, conf, src, extra=None):
        d={'value':value,'confidence':conf if value not in [None,''] else 0.0,'source_text':sanitize_text(src) if src else None}
        if extra: d.update(extra)
        return d
    return {'center_name':field(center,.7,center_src),'national_id':{'value':hash_value(nid),'confidence':.9 if nid else 0.0,'source_text':sanitize_text(nid_src) if nid_src else None},'tracking_number':field(tracking,.85,tracking_src),'date_of_test_or_report':field(date,.9,date_src,{'calendar':'jalali' if date else None}),'patient_name':field(patient_name,.88 if patient_name else 0.0,patient_src),'sex':field(sex,.85 if sex else 0.0,sex_src),'age':field(age,.85 if age is not None else 0.0,age_src),'doctor_name':field(doctor,.75 if doctor else 0.0,doctor_src)}


In [20]:
# =========================
# 8) Lab extraction with stricter local flag handling
# =========================
NUM_RE=re.compile(r'(?<!\d)([*<>]?\d+(?:\.\d+)?)(?!\d)')
UNIT_RE=re.compile(r'(10\^?\s*[36]\s*/?\s*(?:uL|µL|μL)|10\*?[36]\s*/?\s*(?:uL|µL|μL)|mg/dL|mg/dl|g/dL|g/dl|U/L|IU/L|mIU/L|µIU/mL|uIU/mL|ng/mL|ng/ml|pg/ml|IU/L|%|fL|fl|pg|Ratio|Sec|mm/hr)', re.I)
METHOD_RE=re.compile(r'\b(Photometr|ELISA|ECLIA|CLIA|EIA|HPLC|FLC|KIA\s*&\s*ELFA)\b', re.I)

def alias_regex(alias): return re.escape(alias).replace(r'\ ', r'\s+')

def line_is_known_test(line):
    for std,alias in ALL_ALIAS_PHRASES:
        if re.fullmatch(alias_regex(alias), line.strip(), re.I): return std,alias
    return None,None

def find_test_occurrences(text):
    for std,alias in ALL_ALIAS_PHRASES:
        pat=re.compile(rf'(?i)(?<![A-Za-z]){alias_regex(alias)}(?![A-Za-z])')
        for m in pat.finditer(text): yield {'std':std,'alias':alias,'start':m.start(),'end':m.end()}

def next_test_boundary(text,start,min_ahead=5,max_window=240):
    lim=min(len(text), start+max_window); next_pos=lim
    for occ in find_test_occurrences(text[start+min_ahead:lim]):
        pos=start+min_ahead+occ['start']
        if pos>start+min_ahead: next_pos=min(next_pos,pos)
    return next_pos

def extract_flag_from_local_block(block, result_end):
    local=block[result_end:result_end+60]
    local=re.sub(r'(High|Low)\s+risk|High\s*[<>]\s*\d+|Low\s*[<>]\s*\d+', '', local, flags=re.I)
    m=re.search(r'(?<![A-Za-z])(High|Low|Critical)(?![A-Za-z])', local, re.I)
    if m: return m.group(1).capitalize()
    m=re.search(r'(?<![A-Za-z.])(H|L)(?![A-Za-z.])', local)
    if m: return 'High' if m.group(1)=='H' else 'Low'
    if '*' in block[:result_end+20]: return 'rechecked'
    return None

def infer_section(std):
    if std in {'WBC','RBC','HGB','HCT','MCV','MCH','MCHC','RDW-CV','RDW-SD','PLT','PDW','MPV','P-LCR','PCT','NEUT#','LYMPH#','MONO#','EO#','BASO#','NEUT%','LYMPH%','MONO%','EO%','BASO%','ESR'}: return 'hematology'
    if std in {'PT','PT Control','INR','PTT'}: return 'coagulation'
    if std in {'TSH','T3','T4','Free T4','Vitamin D','Vitamin B12','Ferritin','CRP','LH','FSH','Prolactin','Testosterone','Free Testosterone','Estradiol','DHEA-SO4','17OH-Progesterone'}: return 'special_biochemistry_or_hormone'
    if std in {'Urine Culture','Color','Appearance','Specific Gravity','pH','Protein','Urine Glucose','Ketone','Bilirubin Urine','Urobilinogen','Nitrite','Blood/Hb','WBC/HPF','RBC/HPF','Epithelial Cells','Bacteria','Mucus','Casts','Crystals'}: return 'urine_or_microbiology'
    return 'biochemistry'

def parse_test_block(std, alias, block, confidence_base=.70):
    b=block.strip(); b_after=re.sub(rf'(?i)^\s*{alias_regex(alias)}\s*','',b).strip()
    num_match=None
    for m in NUM_RE.finditer(b_after[:120]):
        before=b_after[max(0,m.start()-30):m.start()]
        if GUIDELINE_WORDS_RE.search(before) and m.start()>25: continue
        if re.search(r'\d{4}[/-]\d', b_after[max(0,m.start()-10):m.end()+12]): continue
        num_match=m; break
    if not num_match: return None
    result=num_match.group(1).lstrip('*')
    try: numeric=float(re.sub(r'^[<>]+','',result))
    except Exception: numeric=None
    after=b_after[num_match.end():]
    unit_m=UNIT_RE.search(after[:60]); unit=unit_m.group(0) if unit_m else None
    ref_m=re.search(r'([<>]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?)', after[:90]); ref=re.sub(r'\s+','',ref_m.group(1)) if ref_m else None
    method_m=METHOD_RE.search(after[:120]); method=method_m.group(1) if method_m else None
    flag=extract_flag_from_local_block(b_after, num_match.end())
    return {'section':infer_section(std),'test_name_standard':std,'test_name_raw':alias,'result_value':result,'result_numeric':numeric,'unit':unit,'reference_range':ref,'flag':flag,'method':method,'confidence':confidence_base,'source_text':sanitize_text(b[:300])}

def extract_clean_multiline_blocks(text):
    lines=[l.strip() for l in normalize_text(text).splitlines() if l.strip()]; rows=[]; i=0
    while i<len(lines):
        std,alias=line_is_known_test(lines[i])
        if not std: i+=1; continue
        j=i+1; block=[lines[i]]
        while j<len(lines):
            nstd,_=line_is_known_test(lines[j])
            if nstd: break
            if not re.fullmatch(r'(Test|Result|Unit|Normal Range|Reference Range|Flag|Method|Biochemistry|Hematology)', lines[j], re.I): block.append(lines[j])
            if len(block)>=8: break
            j+=1
        row=parse_test_block(std,alias,'\n'.join(block),.82)
        if row: rows.append(row)
        i=max(j,i+1)
    return rows

def extract_messy_window_rows(text):
    flat=re.sub(r'\s+',' ', normalize_text(text)); rows=[]
    for occ in sorted(find_test_occurrences(flat), key=lambda x:x['start']):
        block=flat[occ['start']:next_test_boundary(flat,occ['start'],max_window=210)]
        row=parse_test_block(occ['std'],occ['alias'],block,.58)
        if row: rows.append(row)
    return rows

def extract_urine_culture(text):
    flat=re.sub(r'\s+',' ', normalize_text(text)); rows=[]
    m=re.search(r'(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?)', flat, re.I)
    if m: rows.append({'section':'microbiology','test_name_standard':'Urine Culture','test_name_raw':'Culture','result_value':m.group(1),'result_numeric':None,'unit':None,'reference_range':None,'flag':None,'method':None,'confidence':.88,'source_text':sanitize_text(flat[max(0,m.start()-80):m.end()+80])})
    pats={'Color':r'Color\s+(Yellow|Amber|Red|Pale|Straw|Yeon|Yetlow)', 'Appearance':r'Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy|Rac)', 'Protein':r'Protein\s+(Negative|Positive|Trace|[0-9+]+)', 'Urine Glucose':r'Glucose\s+(Negative|Positive|Trace|[0-9+]+)', 'Ketone':r'Ketone\s+(Negative|Positive|Trace|[0-9+]+)', 'Nitrite':r'Nitrite\s+(Negative|Positive)', 'Mucus':r'Mucus\s+(Negative|Not seen|Few|Rare|Many)', 'Casts':r'Casts?\s+(Negative|Not seen|Few|Rare|Many)', 'Crystals':r'Crystals?\s+(Negative|Not seen|Few|Rare|Many)', 'Bacteria':r'Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)'}
    for std,pat in pats.items():
        m=re.search(pat,flat,re.I)
        if m: rows.append({'section':'urine_analysis','test_name_standard':std,'test_name_raw':std,'result_value':m.group(1),'result_numeric':None,'unit':None,'reference_range':None,'flag':None,'method':None,'confidence':.66,'source_text':sanitize_text(flat[max(0,m.start()-80):m.end()+80])})
    return rows

def dedup_lab(rows):
    best={}
    for r in rows:
        key=(r['test_name_standard'], str(r.get('result_value')).lower())
        if key not in best or r.get('confidence',0)>best[key].get('confidence',0): best[key]=r
    return list(best.values())

def extract_lab_results(text):
    rows=extract_clean_multiline_blocks(text)+extract_messy_window_rows(text)+extract_urine_culture(text)
    return dedup_lab([r for r in rows if not GUIDELINE_WORDS_RE.fullmatch(str(r.get('test_name_standard','')))])


In [21]:
# =========================
# 9) Build outputs and statuses
# =========================
def extract_radiology(text):
    t=normalize_text(text); rows=[]
    if re.search(r'ultrasound|sonography',t,re.I) or 'سونوگرافی' in t or 'سونوگرافي' in t: rows.append({'part_type':'radiology','section':'radiology','field_name':'imaging_modality','value':'ultrasound','confidence':.8,'source_text':None})
    if 'شکم' in t and 'لگن' in t: rows.append({'part_type':'radiology','section':'radiology','field_name':'body_part_or_exam_name','value':'abdomen and pelvis','confidence':.75,'source_text':None})
    if rows or 'Findings' in t or 'Impression' in t or 'سونو' in t: rows.append({'part_type':'radiology','section':'radiology','field_name':'full_narrative_report','value':t[:4000],'confidence':.65,'source_text':sanitize_text(t[:300])})
    return rows

def common_to_parts(common):
    out=[]
    for k,v in common.items():
        if v.get('value') is not None: out.append({'part_type':'common_field','section':'header','field_name':k,'value':v.get('value'),'result_numeric':None,'unit':None,'reference_range':None,'flag':None,'method':None,'confidence':v.get('confidence',0),'source_text':v.get('source_text')})
    return out

def lab_to_parts(labs):
    return [{'part_type':'lab_result','section':r.get('section'),'field_name':r.get('test_name_standard'),'value':r.get('result_value'),'result_numeric':r.get('result_numeric'),'unit':r.get('unit'),'reference_range':r.get('reference_range'),'flag':r.get('flag'),'method':r.get('method'),'confidence':r.get('confidence'),'source_text':r.get('source_text'),'test_name_raw':r.get('test_name_raw')} for r in labs]

def decide_status(path, quality, ocr, text, medical_ok, doc_type, common, labs, parts):
    tq=ocr.get('text_quality',{})
    if quality.get('status') in ['poor_quality','unreadable'] and path.suffix.lower()!='.pdf': return 'poor_quality_rejected', quality.get('issues',[])
    if not ocr.get('success') or len(text.strip())<CONFIG['min_ocr_text_len']: return 'ocr_failed', ['ocr_text_too_short']
    if tq.get('status') in ['gibberish_text','poor_ocr_text'] and path.suffix.lower()!='.pdf': return 'ocr_unreliable_needs_reupload', [f"ocr_text_quality={tq.get('status')}"]
    if not medical_ok: return 'unrelated_or_not_medical', ['no_medical_signals']
    common_required=sum(1 for k in ['patient_name','date_of_test_or_report','age','sex','national_id','tracking_number'] if common.get(k,{}).get('value'))
    if doc_type=='lab':
        if not labs: return 'partial_extraction',['no_lab_rows_extracted']
        if len(labs)>=3 and common_required>=2: return 'extracted_good',[]
        if len(labs)>=3: return 'needs_manual_review',['missing_or_weak_common_fields']
        return 'needs_manual_review',['low_structured_data']
    if doc_type=='radiology': return ('extracted_good' if parts else 'needs_manual_review'), []
    return 'needs_manual_review',['unknown_document_type']

def make_document_json(index,path,quality,ocr,text,common,labs,parts,status,warnings,doc_type,doc_conf,med_hits,relevance_score,source_preview):
    out_text=sanitize_text(text) if PRIVACY_MODE=='safe_share' else text
    return {'document':{'index':index,'filename':path.name,'source_relative_path':relpath(path),'file_ext':path.suffix.lower(),'mime_type':file_mime_guess(path),'file_size_kb':round(path.stat().st_size/1024,2),'status':status,'warnings':warnings,'document_type':doc_type,'document_type_confidence':doc_conf,'relevance_score':relevance_score,'medical_hits':med_hits,'source_preview':source_preview}, 'quality':quality, 'ocr':{'success':ocr.get('success'),'confidence':ocr.get('confidence'),'text_length':len(text),'source':ocr.get('source'),'best_variant':ocr.get('best_variant'),'best_psm':ocr.get('best_psm'),'best_lang':ocr.get('best_lang'),'text_quality':ocr.get('text_quality'),'candidate_count':ocr.get('candidate_count'),'candidate_scores':ocr.get('candidate_scores',[])[:10]}, 'common_fields':common, 'extracted_data':{'lab_results':labs,'radiology':[p for p in parts if p.get('part_type')=='radiology']}, 'extracted_parts':parts, 'extracted_text':out_text}


In [22]:
# =========================
# 10) Batch processing
# =========================
summary_rows=[]; parts_rows=[]; json_rows=[]; text_rows=[]; documents=[]
for index,path in enumerate(FILES, start=1):
    sid=f'{index:04d}_{safe_id(path)}'; print(f'[{index}/{len(FILES)}] {path.name}')
    try:
        source_preview=create_source_preview(path,sid); quality=assess_file_quality(path); ocr=extract_text_from_file(path,sid); text=normalize_text(ocr.get('text') or '')
        medical_ok,med_hits,relevance_score=is_medical_text(text); doc_type,doc_conf,doc_signals=classify_document(text); common=extract_common_fields(text); labs=extract_lab_results(text) if doc_type=='lab' or medical_ok else []
        parts=common_to_parts(common)+lab_to_parts(labs)
        if doc_type=='radiology': parts += extract_radiology(text)
        status,warnings=decide_status(path,quality,ocr,text,medical_ok,doc_type,common,labs,parts)
        text_path=TEXT_DIR/f'{sid}.txt'; sanitized_text_path=SANITIZED_TEXT_DIR/f'{sid}.txt'; json_path=JSON_DIR/f'{sid}.json'
        text_path.write_text(text, encoding='utf-8'); sanitized_text_path.write_text(sanitize_text(text), encoding='utf-8')
        doc_json=make_document_json(index,path,quality,ocr,text,common,labs,parts,status,warnings,doc_type,doc_conf,med_hits,relevance_score,source_preview)
        json_path.write_text(json.dumps(doc_json, ensure_ascii=False, indent=2), encoding='utf-8'); documents.append(doc_json)
        for r in parts:
            row=dict(r); row.update({'index':index,'filename':path.name,'document_status':status,'document_type':doc_type,'source_relative_path':relpath(path),'text_relative_path':relpath(text_path),'json_relative_path':relpath(json_path)}); parts_rows.append(row)
        summary_rows.append({'index':index,'filename':path.name,'source_relative_path':relpath(path),'file_ext':path.suffix.lower(),'mime_type':file_mime_guess(path),'file_size_kb':round(path.stat().st_size/1024,2),'status':status,'warnings':';'.join(warnings),'quality_status':quality.get('status'),'quality_score':quality.get('overall_quality_score'),'quality_issues':';'.join(quality.get('issues',[])),'ocr_success':ocr.get('success'),'ocr_confidence':round(float(ocr.get('confidence') or 0),3),'ocr_text_length':len(text),'ocr_text_quality_status':ocr.get('text_quality',{}).get('status'),'ocr_text_quality_score':ocr.get('text_quality',{}).get('score'),'ocr_source':ocr.get('source'),'best_ocr_variant':ocr.get('best_variant'),'best_psm':ocr.get('best_psm'),'medical_ok':medical_ok,'relevance_score':relevance_score,'document_type':doc_type,'document_type_confidence':doc_conf,'common_field_count':sum(1 for v in common.values() if v.get('value') is not None),'lab_result_count':len(labs),'patient_name_found':common.get('patient_name',{}).get('value') is not None,'date_found':common.get('date_of_test_or_report',{}).get('value') is not None,'age_found':common.get('age',{}).get('value') is not None,'sex_found':common.get('sex',{}).get('value') is not None,'doctor_found':common.get('doctor_name',{}).get('value') is not None,'text_relative_path':relpath(text_path),'json_relative_path':relpath(json_path),'source_preview':source_preview})
        json_rows.append({'index':index,'filename':path.name,'status':status,'document_type':doc_type,'quality_json':json.dumps(quality,ensure_ascii=False),'ocr_json':json.dumps({k:v for k,v in doc_json['ocr'].items() if k!='candidate_scores'},ensure_ascii=False),'common_fields_json':json.dumps(common,ensure_ascii=False),'lab_results_json':json.dumps(labs,ensure_ascii=False),'extracted_parts_json':json.dumps(parts,ensure_ascii=False),'document_json':json.dumps(doc_json,ensure_ascii=False),'extracted_text':sanitize_text(text) if PRIVACY_MODE=='safe_share' else text,'source_relative_path':relpath(path),'text_relative_path':relpath(text_path),'json_relative_path':relpath(json_path)})
        text_rows.append({'index':index,'filename':path.name,'source_relative_path':relpath(path),'text_relative_path':relpath(text_path),'sanitized_text_relative_path':relpath(sanitized_text_path),'extracted_text':sanitize_text(text) if PRIVACY_MODE=='safe_share' else text,'sanitized_text':sanitize_text(text)})
    except Exception as e:
        traceback.print_exc(); summary_rows.append({'index':index,'filename':path.name,'source_relative_path':relpath(path),'status':'notebook_error','warnings':str(e),'lab_result_count':0})
summary_df=pd.DataFrame(summary_rows); parts_df=pd.DataFrame(parts_rows); json_rows_df=pd.DataFrame(json_rows); texts_df=pd.DataFrame(text_rows)
summary_path=OUTPUT_DIR/'document_summary.csv'; parts_path=OUTPUT_DIR/'extracted_parts.csv'; json_rows_path=OUTPUT_DIR/'document_json_rows.csv'; texts_path=OUTPUT_DIR/'extracted_texts.csv'; jsonl_path=OUTPUT_DIR/'documents.jsonl'
summary_df.to_csv(summary_path,index=False,encoding='utf-8-sig'); parts_df.to_csv(parts_path,index=False,encoding='utf-8-sig'); json_rows_df.to_csv(json_rows_path,index=False,encoding='utf-8-sig'); texts_df.to_csv(texts_path,index=False,encoding='utf-8-sig')
with jsonl_path.open('w',encoding='utf-8') as f:
    for d in documents: f.write(json.dumps(d,ensure_ascii=False)+'\n')
print('Saved outputs in', relpath(OUTPUT_DIR))
display(summary_df['status'].value_counts().to_frame('count'))
display(summary_df)
display(parts_df.head(100))


[1/20] -2147483648_-210193.jpg
[2/20] -2147483648_-210195.jpg
[3/20] 0014161672_14041209_O_00404121721.pdf
[4/20] 0020139871_14041209_O_00404121714.pdf
[5/20] 0021858845_14041209_O_00404121731.pdf
[6/20] 0023471026_14041209_O_00404121728.pdf
[7/20] 0024150010_14041209_O_00404121722.pdf
[8/20] 0025631314_14041209_O_00404121726.pdf
[9/20] 0025692283_14041209_O_00404121730.pdf
[10/20] 20260427_181554.jpg
[11/20] 20260427_181636.jpg
[12/20] 20260427_181654.jpg
[13/20] 20260427_181713.jpg
[14/20] 20260427_181919.jpg
[15/20] ۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg
[16/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg
[17/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg
[18/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg
[19/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۲۰.jpg
[20/20] ۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg
Saved outputs in notebook_outputs/selected_samples_v6


,count
status,
needs_manual_review,11
extracted_good,7
partial_extraction,1
ocr_unreliable_needs_reupload,1


,index,filename,source_relative_path,file_ext,mime_type,file_size_kb,status,warnings,quality_status,quality_score,...,common_field_count,lab_result_count,patient_name_found,date_found,age_found,sex_found,doctor_found,text_relative_path,json_relative_path,source_preview
0,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,.jpg,image/jpeg,77.79,needs_manual_review,missing_or_weak_common_fields,good_quality,0.633,...,0,11,False,False,False,False,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...,notebook_outputs/selected_samples_v6/source_pr...
1,2,-2147483648_-210195.jpg,samples/selected samples/-2147483648_-210195.jpg,.jpg,image/jpeg,42.00,partial_extraction,no_lab_rows_extracted,good_quality,0.612,...,1,0,False,False,False,False,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0002...,notebook_outputs/selected_samples_v6/source_pr...
2,3,0014161672_14041209_O_00404121721.pdf,samples/selected samples/0014161672_14041209_O...,.pdf,application/pdf,44.22,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,22,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0003...,notebook_outputs/selected_samples_v6/source_pr...
3,4,0020139871_14041209_O_00404121714.pdf,samples/selected samples/0020139871_14041209_O...,.pdf,application/pdf,41.35,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,4,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0004...,notebook_outputs/selected_samples_v6/source_pr...
4,5,0021858845_14041209_O_00404121731.pdf,samples/selected samples/0021858845_14041209_O...,.pdf,application/pdf,42.88,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,17,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0005...,notebook_outputs/selected_samples_v6/source_pr...
5,6,0023471026_14041209_O_00404121728.pdf,samples/selected samples/0023471026_14041209_O...,.pdf,application/pdf,40.79,needs_manual_review,low_structured_data,pdf_quality_not_assessed_here,NaN,...,7,2,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0006...,notebook_outputs/selected_samples_v6/source_pr...
6,7,0024150010_14041209_O_00404121722.pdf,samples/selected samples/0024150010_14041209_O...,.pdf,application/pdf,44.13,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,22,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0007...,notebook_outputs/selected_samples_v6/source_pr...
7,8,0025631314_14041209_O_00404121726.pdf,samples/selected samples/0025631314_14041209_O...,.pdf,application/pdf,41.18,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,4,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0008...,notebook_outputs/selected_samples_v6/source_pr...
8,9,0025692283_14041209_O_00404121730.pdf,samples/selected samples/0025692283_14041209_O...,.pdf,application/pdf,42.40,extracted_good,,pdf_quality_not_assessed_here,NaN,...,7,4,True,True,True,True,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0009...,notebook_outputs/selected_samples_v6/source_pr...
9,10,20260427_181554.jpg,samples/selected samples/20260427_181554.jpg,.jpg,image/jpeg,42.47,needs_manual_review,missing_or_weak_common_fields,good_quality,0.746,...,2,6,False,False,False,False,True,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0010...,notebook_outputs/selected_samples_v6/source_pr...


,part_type,section,field_name,value,result_numeric,unit,reference_range,flag,method,confidence,source_text,test_name_raw,index,filename,document_status,document_type,source_relative_path,text_relative_path,json_relative_path
0,lab_result,biochemistry,AST,9.0,9.00,NaN,NaN,NaN,NaN,0.82,ast\ngit\ncua\nzz\nnH\n9.0\n1s\n255,AST,1,-2147483648_-210193.jpg,needs_manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
1,lab_result,special_biochemistry_or_hormone,Testosterone,020.93,20.93,NaN,NaN,NaN,NaN,0.82,Testosterone\nas\nHA\n020.93,Testosterone,1,-2147483648_-210193.jpg,needs_manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
2,lab_result,special_biochemistry_or_hormone,Free Testosterone,30,30.00,NaN,NaN,NaN,NaN,0.82,Free Testosterone\na\npa\nza.\narly fellctar 3...,Free Testosterone,1,-2147483648_-210193.jpg,needs_manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
3,lab_result,special_biochemistry_or_hormone,Estradiol,209,209.00,NaN,NaN,NaN,NaN,0.82,Estradiol\nppt\n209\naes:\nostmenoeasal untren...,Estradiol,1,-2147483648_-210193.jpg,needs_manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
4,lab_result,special_biochemistry_or_hormone,Ferritin,4970,4970.00,NaN,NaN,NaN,NaN,0.82,Ferritin\n4970\ngmt\ncua\ndoze\nbetow\nurine a...,Ferritin,1,-2147483648_-210193.jpg,needs_manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,lab_result,hematology,HCT,45.9,45.90,%,38.8-50,NaN,NaN,0.82,H.C.T\n45.9\n%\n38.8-50 %,H.C.T,7,0024150010_14041209_O_00404121722.pdf,extracted_good,lab,samples/selected samples/0024150010_14041209_O...,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0007...
96,lab_result,hematology,MCV,84.7,84.70,fL,81.2-95.1,NaN,NaN,0.82,M.C.V\n84.7\nfL\n81.2-95.1 fl,M.C.V,7,0024150010_14041209_O_00404121722.pdf,extracted_good,lab,samples/selected samples/0024150010_14041209_O...,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0007...
97,lab_result,hematology,MCH,28.2,28.20,pg,25.8-33.1,NaN,NaN,0.82,M.C.H\n28.2\npg\n25.8-33.1 pg,M.C.H,7,0024150010_14041209_O_00404121722.pdf,extracted_good,lab,samples/selected samples/0024150010_14041209_O...,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0007...
98,lab_result,hematology,MCHC,33.3,33.30,g/dL,32-36,NaN,NaN,0.82,M.C.H.C\n33.3\ng/dL\n32-36 g/dl,M.C.H.C,7,0024150010_14041209_O_00404121722.pdf,extracted_good,lab,samples/selected samples/0024150010_14041209_O...,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0007...


In [23]:
# =========================
# 11) Clean comparison HTML report
# =========================
def df_to_html_table(df, columns=None, max_rows=120):
    if df is None or df.empty: return '<p><em>No rows.</em></p>'
    if columns: df = df[[c for c in columns if c in df.columns]]
    if len(df)>max_rows: df=df.head(max_rows)
    return df.to_html(index=False, escape=True)

sections=[]
for d in documents:
    idx=d['document']['index']; fn=d['document']['filename']; status=d['document']['status']; preview=d['document'].get('source_preview')
    sub=parts_df[parts_df['index']==idx].copy(); common_sub=sub[sub['part_type']=='common_field']; labs_sub=sub[sub['part_type']=='lab_result']; text=d.get('extracted_text','')
    color='#e8f5e9' if status=='extracted_good' else ('#fff8e1' if 'review' in status or 'partial' in status else '#ffebee')
    preview_html=f'<img src="../../{html.escape(preview)}" style="max-width:700px;border:1px solid #ddd"/>' if preview else ''
    sections.append(f"""<section style="background:{color};padding:16px;margin:20px 0;border-radius:8px"><h2>{idx}. {html.escape(fn)}</h2><p><b>Status:</b> {html.escape(status)} | <b>Type:</b> {html.escape(str(d['document'].get('document_type')))} | <b>OCR text quality:</b> {html.escape(str(d['ocr'].get('text_quality',{}).get('status')))} | <b>OCR len:</b> {len(text)} | <b>Lab rows:</b> {len(d['extracted_data'].get('lab_results', []))}</p><details open><summary><b>Source preview</b></summary>{preview_html}</details><details open><summary><b>Common fields</b></summary>{df_to_html_table(common_sub, ['field_name','value','confidence','source_text'])}</details><details open><summary><b>Lab / extracted rows</b></summary>{df_to_html_table(labs_sub, ['field_name','value','unit','reference_range','flag','confidence','source_text'])}</details><details><summary><b>Exact extracted text</b></summary><pre style="white-space:pre-wrap;background:white;padding:10px;border:1px solid #ddd">{html.escape(text[:8000])}</pre></details></section>""")
html_doc=f"""<!doctype html><html><head><meta charset="utf-8"/><title>Selected Samples Extraction Review v6</title><style>body{{font-family:Arial,sans-serif;margin:24px;line-height:1.4}}table{{border-collapse:collapse;font-size:13px;width:100%;background:white}}td,th{{border:1px solid #ddd;padding:6px;vertical-align:top}}th{{background:#f2f2f2}}pre{{direction:ltr}}</style></head><body><h1>Selected Samples Extraction Review v6</h1><p>Generated from notebook. Paths are relative to repository.</p><h2>Summary</h2>{summary_df.to_html(index=False,escape=True)}{''.join(sections)}</body></html>"""
review_path=OUTPUT_DIR/'review_compare.html'; review_path.write_text(html_doc, encoding='utf-8')
print('Saved HTML review:', relpath(review_path))


Saved HTML review: notebook_outputs/selected_samples_v6/review_compare.html


In [24]:
# =========================
# 12) Weak files and suspicious flags
# =========================
weak_df = summary_df[summary_df['status'] != 'extracted_good'].copy()
print('Weak/non-good files')
display(weak_df[[c for c in ['index','filename','status','warnings','quality_status','quality_issues','ocr_text_quality_status','ocr_text_quality_score','ocr_text_length','best_ocr_variant','best_psm','document_type','lab_result_count','patient_name_found','date_found','age_found','sex_found','text_relative_path','json_relative_path'] if c in weak_df.columns]])
print('Rows with abnormal flags for review')
if not parts_df.empty and 'flag' in parts_df.columns:
    suspect = parts_df[(parts_df['part_type']=='lab_result') & parts_df['flag'].notna() & (parts_df['flag'].astype(str).str.len()>0)]
    display(suspect[[c for c in ['index','filename','field_name','value','reference_range','flag','source_text'] if c in suspect.columns]].head(200))


Weak/non-good files


,index,filename,status,warnings,quality_status,quality_issues,ocr_text_quality_status,ocr_text_quality_score,ocr_text_length,best_ocr_variant,best_psm,document_type,lab_result_count,patient_name_found,date_found,age_found,sex_found,text_relative_path,json_relative_path
0,1,-2147483648_-210193.jpg,needs_manual_review,missing_or_weak_common_fields,good_quality,too_bright;possible_rotation_or_sideways,good_text,0.639,1475,light_sharpen,11.0,lab,11,False,False,False,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0001...
1,2,-2147483648_-210195.jpg,partial_extraction,no_lab_rows_extracted,good_quality,too_bright;possible_rotation_or_sideways,usable_noisy_text,0.315,424,light_autocontrast,11.0,lab,0,False,False,False,False,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0002...
5,6,0023471026_14041209_O_00404121728.pdf,needs_manual_review,low_structured_data,pdf_quality_not_assessed_here,,usable_noisy_text,0.434,406,pdf_text_layer,NaN,lab,2,True,True,True,True,notebook_outputs/selected_samples_v6/texts/000...,notebook_outputs/selected_samples_v6/json/0006...
9,10,20260427_181554.jpg,needs_manual_review,missing_or_weak_common_fields,good_quality,,usable_noisy_text,0.435,344,light_autocontrast_rot90,6.0,lab,6,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0010...
10,11,20260427_181636.jpg,needs_manual_review,missing_or_weak_common_fields,good_quality,,usable_noisy_text,0.327,310,mild_contrast_rot90,6.0,lab,4,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0011...
11,12,20260427_181654.jpg,needs_manual_review,low_structured_data,needs_preprocessing,,usable_noisy_text,0.343,296,mild_contrast_rot90,6.0,lab,2,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0012...
12,13,20260427_181713.jpg,ocr_unreliable_needs_reupload,ocr_text_quality=gibberish_text,good_quality,,gibberish_text,0.202,199,original_oriented_rot90,11.0,lab,2,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0013...
13,14,20260427_181919.jpg,needs_manual_review,missing_or_weak_common_fields,good_quality,,good_text,0.598,437,original_oriented_rot90,11.0,lab,6,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0014...
15,16,۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg,needs_manual_review,missing_or_weak_common_fields,good_quality,,good_text,0.824,1210,mild_contrast,11.0,lab,15,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0016...
16,17,۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg,needs_manual_review,unknown_document_type,needs_preprocessing,severe_blur,usable_noisy_text,0.345,1147,threshold_fallback,6.0,unknown_medical,1,False,False,False,False,notebook_outputs/selected_samples_v6/texts/001...,notebook_outputs/selected_samples_v6/json/0017...


Rows with abnormal flags for review


,index,filename,field_name,value,reference_range,flag,source_text
31,3,0014161672_14041209_O_00404121721.pdf,LYMPH%,43,20-40,High,Lymphocytes\n43\n%\n20 - 40\nHigh
36,3,0014161672_14041209_O_00404121721.pdf,AST,19,<37,Low,SGOT(AST)\n19\nU/L\n<37\nPhotometr
37,3,0014161672_14041209_O_00404121721.pdf,ALT,21,<40,Low,SGPT(ALT)\n21\nU/L\n< 40\nPhotometr\nThis answ...
49,4,0020139871_14041209_O_00404121714.pdf,AST,47,<37,High,SGOT(AST)\n47\nU/L\n<37\nHigh\n*\nPhotometr
50,4,0020139871_14041209_O_00404121714.pdf,ALT,46,<40,High,SGPT(ALT)\n46\nU/L\n< 40\nHigh\n*\nPhotometr\n...
71,5,0021858845_14041209_O_00404121731.pdf,LYMPH%,17.7,20-40,Low,Lymphocytes\n17.7\n%\n20 - 40\nLow
109,7,0024150010_14041209_O_00404121722.pdf,AST,20,<37,Low,SGOT(AST)\n20\nU/L\n<37\nPhotometr
110,7,0024150010_14041209_O_00404121722.pdf,ALT,22,<40,Low,SGPT(ALT)\n22\nU/L\n< 40\nPhotometr\nThis answ...
122,8,0025631314_14041209_O_00404121726.pdf,AST,18,<37,Low,SGOT(AST)\n18\nU/L\n<37\nPhotometr
123,8,0025631314_14041209_O_00404121726.pdf,ALT,14,<40,Low,SGPT(ALT)\n14\nU/L\n< 40\nPhotometr\nThis answ...
